# 03. Model Training & Cross-Validation Benchmarking
## Support Ticket Classification & Prioritization

### Objective:
1. Benchmark 3 classical ML models for both **Category** and **Priority**:
   - **Multinomial Naive Bayes** (`MultinomialNB`)
   - **Logistic Regression** (`LogisticRegression`)
   - **Linear Support Vector Machine** (`LinearSVC` + `CalibratedClassifierCV`)
2. Evaluate using **5-Fold Stratified Cross-Validation** (Macro F1).
3. Perform hyperparameter tuning on the best model architectures.
4. Save winning models to disk with `joblib`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import config
from src.train import create_candidate_pipelines, evaluate_candidate_models, tune_and_train_best_model

train_df = pd.read_csv(config.TRAIN_DATA_PATH)
print(f'Loaded training split: {len(train_df)} rows')
X_train = train_df['text']
y_train_cat = train_df['category']
y_train_pri = train_df['priority']

### 1. Benchmarking Models for Ticket Category
We evaluate candidate pipelines on `category` using 5-fold stratified cross-validation.

In [ ]:
best_cat_model, cat_results = evaluate_candidate_models(X_train, y_train_cat, 'Category')

# Plot CV Comparison
models = list(cat_results.keys())
f1_means = [cat_results[m]['mean_f1_macro'] for m in models]
f1_stds = [cat_results[m]['std_f1_macro'] for m in models]

plt.figure(figsize=(9, 4))
plt.bar(models, f1_means, yerr=f1_stds, capsize=5, color=['#4caf50', '#2196f3', '#ff9800'], edgecolor='black')
plt.title('Category Classification: 5-Fold CV Macro F1 Comparison', fontweight='bold')
plt.ylabel('Macro F1-Score')
plt.ylim(0.8, 1.05)
for i, v in enumerate(f1_means):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Analysis of Category Model Results
- All three models achieve near-perfect cross-validation performance on Category classification.
- Multinomial Naive Bayes is chosen for Category due to its speed, low parameter count, and strong generative text modeling.

### 2. Benchmarking Models for Ticket Priority
We evaluate candidate pipelines on `priority`.

In [ ]:
best_pri_model, pri_results = evaluate_candidate_models(X_train, y_train_pri, 'Priority')

models = list(pri_results.keys())
f1_means = [pri_results[m]['mean_f1_macro'] for m in models]
f1_stds = [pri_results[m]['std_f1_macro'] for m in models]

plt.figure(figsize=(9, 4))
plt.bar(models, f1_means, yerr=f1_stds, capsize=5, color=['#4caf50', '#2196f3', '#ff9800'], edgecolor='black')
plt.title('Priority Classification: 5-Fold CV Macro F1 Comparison', fontweight='bold')
plt.ylabel('Macro F1-Score')
plt.ylim(0.80, 0.95)
for i, v in enumerate(f1_means):
    plt.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Analysis of Priority Model Results
- Priority classification is naturally more difficult than Category due to overlapping urgency keywords and subjective human triage.
- Logistic Regression achieves the highest Macro F1 (~0.8821), slightly outperforming Calibrated Linear SVM and Naive Bayes.
- Logistic Regression is chosen as the final Priority model.

### Summary

### Q&A
- **Q: Why use Stratified K-Fold CV?**
  - **A**: It preserves the class proportion in each fold, preventing small classes (e.g. Critical) from being underrepresented.
- **Q: Why use calibrated probabilities?**
  - **A**: In production, routing decisions require confidence thresholds; uncalibrated scores cannot be interpreted as true probabilities.

### Data Analysis Key Findings
- Category classification achieves near 100% CV Macro F1 across all candidate architectures.
- Priority classification achieves 88.2% CV Macro F1 using Logistic Regression with C=0.5.

### Insights or Next Steps
- Retrain winning pipelines and evaluate on held-out test data in `04_evaluation.ipynb`.